Vision Segmentation: UNet, Mask2Former & Promptable SAM

Image segmentation partitions an image into semantically coherent pixel groups. Over the past decade,
the field has progressed through four major paradigm shifts:
1. Per-pixel dense classification with encoder-decoder skip connections (UNet).
2. Top-down detect-then-segment with bounding box proposals (Mask R-CNN).
3. Universal query-based segmentation with masked cross-attention (Mask2Former).
4. Promptable foundation models with random Fourier feature prompt embeddings (SAM).

## 2. Theoretical Foundations & Mathematical Formulations

- **Soft Dice Loss**:
  $$\mathcal{L}_{\text{Dice}} = 1 - \frac{2 \sum_i p_i t_i + \epsilon}{\sum_i p_i^2 + \sum_i t_i^2 + \epsilon}$$
- **Mask2Former Masked Attention**:
  $$\operatorname{MaskedAttn}(Q, K, V) = \operatorname{softmax}\left( \frac{Q K^T}{\sqrt{D}} + \mathcal{M} \right) V$$
- **SAM Random Gaussian Fourier Prompt Encoding**:
  $$\gamma(p) = \left[ \sin(2\pi \cdot p B^T), \ \cos(2\pi \cdot p B^T) \right] \in \mathbb{R}^{2 D_{\text{feat}}}$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "02_vision_segmentation_unet_mask2former_and_sam_promptable.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# Soft Dice Loss implementation
def dice_loss(pred_mask: torch.Tensor, target_mask: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    intersection = torch.sum(pred_mask * target_mask, dim=(-2, -1))
    cardinality = torch.sum(pred_mask**2, dim=(-2, -1)) + torch.sum(target_mask**2, dim=(-2, -1))
    dice = (2.0 * intersection + eps) / (cardinality + eps)
    return 1.0 - dice

dice_calc = dice_loss(golden["pred_mask"], golden["target_mask"])
diff_dice = torch.max(torch.abs(dice_calc - golden["dice_loss_ref"])).item()
print(f"Dice Loss max abs error: {diff_dice:.2e}")
assert diff_dice < 1e-5

In [ ]:
# Mask2Former Masked Cross-Attention
def mask2former_masked_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    attn_mask: torch.Tensor,
) -> torch.Tensor:
    d = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d)
    masked_scores = scores + attn_mask
    probs = torch.softmax(masked_scores, dim=-1)
    return probs @ v

m2f_calc = mask2former_masked_attention(golden["q"], golden["k"], golden["v"], golden["attn_mask"])
diff_m2f = torch.max(torch.abs(m2f_calc - golden["m2f_out"])).item()
print(f"Mask2Former max abs error: {diff_m2f:.2e}")
assert diff_m2f < 1e-5

In [ ]:
# SAM Random Gaussian Fourier Prompt Encoding
def sam_fourier_prompt_embed(points: torch.Tensor, gaussian_matrix: torch.Tensor) -> torch.Tensor:
    proj = 2.0 * math.pi * (points @ gaussian_matrix.t())
    return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

sam_calc = sam_fourier_prompt_embed(golden["points"], golden["gaussian_matrix"])
diff_sam = torch.max(torch.abs(sam_calc - golden["sam_prompt_embed"])).item()
print(f"SAM Fourier Embed max abs error: {diff_sam:.2e}")
assert diff_sam < 1e-5

In [ ]:
# Verification Summary and Invariant Metrics
print("=" * 60)
print(f"{'Operator':<30} | {'Max Abs Error':<15} | Status")
print("-" * 60)
print(f"{'Soft Dice Loss':<30} | {diff_dice:<15.2e} | PASSED")
print(f"{'Mask2Former Masked Attention':<30} | {diff_m2f:<15.2e} | PASSED")
print(f"{'SAM Fourier Prompt Embed':<30} | {diff_sam:<15.2e} | PASSED")
print("=" * 60)

## 8. Summary, Invariants & Diagnostic Checklist

- [x] Soft Dice Loss provides smooth, scale-invariant gradients in extreme class imbalance.
- [x] Mask2Former restricts cross-attention to predicted foreground regions, accelerating training.
- [x] SAM Random Gaussian Fourier features project 2D prompt coordinates into continuous high-dimensional embeddings.
- [x] Numerical parity verified across PyTorch, JAX, Candle, and WGSL shaders within $< 10^{-5}$ tolerance.